In [1]:
import duckdb
from functions.evaluation import evaluate
from networks.cnn_large_batchnorm import CNNModel
from networks.cnn_network import build_dataloaders


con = duckdb.connect('../capillary.db')
df = con.execute(""" 
                 SELECT row_id, value,fractions, boundaries,albumin,antitrypsin,orosomukoid,haptoglobin,crp,igg,iga,igm, label, set, interpretation
                 FROM protein_data
                 WHERE value IS NOT NULL
                 AND observation_nr = 1
                 AND analysis IS NOT NULL
                 AND protein_value IS NOT NULL
                 """).df()
con.close()


train_rows = df[df['set'] == 'train']
val_rows   = df[df['set'] == 'val']
test_rows  = df[df['set'] == 'test']

drop_indices = train_rows[train_rows['label'] == 0].sample(frac=0.7).index


train_rows = train_rows.drop(drop_indices)
train_rows = train_rows[train_rows['label'].isin([0,1])]
val_rows = val_rows[val_rows['label'].isin([0,1])]

print(f"Antal utan m-komponent i träningsdatan: {len(train_rows[train_rows['label'] == 0])}")
print(f"Antal med m-komponent i träningsdatan: {len(train_rows[train_rows['label'] == 1])}")

CNN = CNNModel()
cnn_train_dl, cnn_val_dl, _ = build_dataloaders(train_rows, val_rows, val_rows)
CNN.reset_weights()
CNN.retrain(cnn_train_dl,cnn_val_dl,patience=15)

Antal utan m-komponent i träningsdatan: 17903
Antal med m-komponent i träningsdatan: 2553
Total parameters: 236,066
  -> ny bästa modell sparad till ../models/convolution_model_larger.pth
Epoch   0 | train: 0.5087 | val: 0.3677 | acc: 96.90% | AUC: 0.937  | LR: 0.001
  -> ny bästa modell sparad till ../models/convolution_model_larger.pth
Epoch   1 | train: 0.2878 | val: 0.1567 | acc: 98.54% | AUC: 0.982  | LR: 0.001
Epoch   2 | train: 0.2303 | val: 0.6526 | acc: 57.10% | AUC: 0.978  | LR: 0.001
  -> ny bästa modell sparad till ../models/convolution_model_larger.pth
Epoch   3 | train: 0.1837 | val: 0.1142 | acc: 97.90% | AUC: 0.985  | LR: 0.001
  -> ny bästa modell sparad till ../models/convolution_model_larger.pth
Epoch   4 | train: 0.1724 | val: 0.2833 | acc: 87.10% | AUC: 0.989  | LR: 0.001
Epoch   5 | train: 0.1784 | val: 0.1914 | acc: 93.25% | AUC: 0.985  | LR: 0.001
  -> ny bästa modell sparad till ../models/convolution_model_larger.pth
Epoch   6 | train: 0.1520 | val: 0.1494 | ac

In [2]:
con = duckdb.connect('../capillary.db')
df = con.execute(""" 
                 SELECT row_id, value,fractions, boundaries,albumin,antitrypsin,orosomukoid,haptoglobin,crp,igg,iga,igm, label, set, interpretation
                 FROM protein_data
                 WHERE value IS NOT NULL
                 AND observation_nr = 1
                 AND analysis IS NOT NULL
                 AND protein_value IS NOT NULL
                 """).df()
con.close()


k_fold_rows = df[df['set'].isin(['train','val']) ].copy()
k_fold_rows = k_fold_rows[k_fold_rows['label'].isin([0,1])]
drop_indices = k_fold_rows[k_fold_rows['label'] == 0].sample(frac=0.7).index
k_fold_rows = k_fold_rows.drop(drop_indices)
print(f"Totalt antal utan m-komponent i K-fold poolen: {len(k_fold_rows[k_fold_rows['label'] == 0])}")
print(f"Totalt antal med m-komponent i K-fold poolen: {len(k_fold_rows[k_fold_rows['label'] == 1])}")

test_rows  = df[df['set'] == 'test']


CNN = CNNModel()
CNN.retrain_with_k_fold(k_fold_rows)

Totalt antal utan m-komponent i K-fold poolen: 18848
Totalt antal med m-komponent i K-fold poolen: 2692
Total parameters: 236,066
--- Startar 10-Fold Cross Validation ---

 FOLD 1/10
  -> ny bästa modell sparad till ../models/convolution_model_larger_fold1.pth
Epoch   0 | train: 0.5489 | val: 0.7589 | acc: 90.80% | AUC: 0.882  | LR: 0.001
  -> ny bästa modell sparad till ../models/convolution_model_larger_fold1.pth
Epoch   1 | train: 0.3332 | val: 0.4672 | acc: 94.42% | AUC: 0.958  | LR: 0.001
  -> ny bästa modell sparad till ../models/convolution_model_larger_fold1.pth
Epoch   2 | train: 0.2554 | val: 0.2367 | acc: 91.36% | AUC: 0.971  | LR: 0.001
  -> ny bästa modell sparad till ../models/convolution_model_larger_fold1.pth
Epoch   3 | train: 0.2112 | val: 0.2989 | acc: 96.75% | AUC: 0.975  | LR: 0.001
Epoch   4 | train: 0.2002 | val: 0.2256 | acc: 93.96% | AUC: 0.973  | LR: 0.001
  -> ny bästa modell sparad till ../models/convolution_model_larger_fold1.pth
Epoch   5 | train: 0.1856 |

,fold,train_loss,val_loss,val_accuracy,val_auc,val_spec,val_sens,tn,fp,fn,tp
0,1,0.084749,0.215014,97.815985,0.987984,0.987248,0.914815,1858,24,23,247
1,2,0.215133,0.233314,83.364312,0.991918,0.811371,0.988889,1527,355,3,267
2,3,0.083170,0.145702,96.934510,0.986346,0.977176,0.914498,1841,43,23,246
3,4,0.122147,0.149107,94.893222,0.989922,0.944828,0.977695,1781,104,6,263
4,5,0.056916,0.132144,96.468401,0.992449,0.969198,0.933086,1825,58,18,251
5,6,0.118712,0.127925,92.711235,0.993871,0.919363,0.981413,1733,152,5,264
6,7,0.056483,0.118050,96.655829,0.989807,0.970276,0.940520,1828,56,16,253
7,8,0.095167,0.245244,96.748723,0.982865,0.981953,0.866171,1850,34,36,233
8,9,0.076433,0.179136,96.843083,0.985903,0.973475,0.933086,1835,50,18,251
9,10,0.108107,0.116741,93.871866,0.993520,0.934218,0.970260,1761,124,8,261


In [3]:
test_rows = test_rows[test_rows['label'].isin([0,1])]
result = CNN.predict(test_rows)
_ = evaluate(result,threshold=0.5,proportion=70)

TypeError: evaluate() got an unexpected keyword argument 'threshold'